In [24]:
import boto3, joblib, io, json
import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [25]:
BUCKET = 'spam-classifier-mlops-yt2'  



# s3 = boto3.client('s3')

# obj = s3.get_object(Bucket=BUCKET, Key='data/train/sms_spam.csv')
# df = pd.read_csv(io.BytesIO(obj['Body'].read()), 
#                  sep=',',
#                  header=0)  # ← has a header row

# # Clean
# df = df.dropna(subset=['text', 'label'])
# df['text'] = df['text'].astype(str)

# print(df.shape)
# print(df['label'].value_counts())
# print(df.head())

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=BUCKET, Key='data/train/sms_spam.csv')
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

# Clean
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)

print(df.shape)
print(df['label'].value_counts())
print(df.head())

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [26]:
obj2 = s3.get_object(Bucket=BUCKET, Key='data/train/sms_spam.csv')
raw_text = obj2['Body'].read().decode('utf-8')
print("Total characters:", len(raw_text))
print("First 300 chars:")
print(raw_text[:300])

Total characters: 480099
First 300 chars:
label,text
ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
ham,Ok lar... Joking wif u oni...
spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's ap


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', 
                              max_features=5000, 
                              ngram_range=(1, 2))),
    ('clf', MultinomialNB(alpha=0.1))
])

model.fit(X_train, y_train)
print("Training done!")

Training done!


In [28]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.98      0.94      0.96       149

    accuracy                           0.99      1115
   macro avg       0.98      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115

[[963   3]
 [  9 140]]


In [30]:
# # Save baseline confidence stats
# baseline_probs = model.predict_proba(X_test).max(axis=1)
# baseline_stats = {
#     'mean_confidence': float(np.mean(baseline_probs)),
#     'std_confidence':  float(np.std(baseline_probs)),
#     'min_confidence':  float(np.min(baseline_probs)),
#     'model_version':   'v1'
# }
# print('Baseline stats:', baseline_stats)

# # Save model
# buf = io.BytesIO()
# joblib.dump(model, buf)
# buf.seek(0)
# s3.put_object(Bucket=BUCKET, Key='models/naive_bayes_v1.pkl', Body=buf)

# # Save baseline stats
# s3.put_object(
#     Bucket=BUCKET,
#     Key='models/baseline_stats_v1.json',
#     Body=json.dumps(baseline_stats)
# )

# print("✅ Model saved to S3!")import pickle, boto3, io

import pickle, boto3, io

buf = io.BytesIO()
pickle.dump(model, buf)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key='models/naive_bayes_v1.pkl', Body=buf)
print("Done!")

Done!
